# Celeb-DF-v1 Test Embeddings

Notebook này generate CLIP image embeddings cho Celeb-DF-v1 với 3 setting thứ tự:

- `sequential`: giữ đúng thứ tự sau khi filter từ CSV.
- `shuffle`: shuffle deterministic rồi generate.
- `balanced_16_16`: lấy batch logic 16 REAL + 16 FAKE liên tục, dừng khi hết REAL.

Output `.pt` lưu `features`, `labels`, `paths`, và metadata `order_setting` để eval/check lại thứ tự.

In [1]:
from pathlib import Path

import open_clip
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

ModuleNotFoundError: No module named 'open_clip'

In [ ]:
# Kaggle paths. Chỉnh lại nếu dataset mount khác.
CSV_PATH = "/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv"
DEEPFAKEBENCH_ROOT = "/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench"
CELEBDFV1_ORIGINAL_ROOT = f"{DEEPFAKEBENCH_ROOT}/Celeb-DF-v1"

# Chạy level nào thì sửa list này. Để [1, 2, 3, 4, 5] là gen toàn bộ 5 level.
LEVELS = [1, 2, 3, 4, 5]

# Kaggle có thể mount theo /kaggle/input/datasets/{owner}/{slug} hoặc /kaggle/input/{slug}.
# Mapping dưới đây ưu tiên đúng owner/slug từ URL bạn đưa, rồi fallback qua slug ngắn.
LEVEL_DATASET_ROOTS = {
    1: [
        Path("/kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output"),
        Path("/kaggle/input/celebdfv1-level-1/processed_output"),
    ],
    2: [
        Path("/kaggle/input/datasets/elisevo/celebdfv1-level-2/processed_output"),
        Path("/kaggle/input/celebdfv1-level-2/processed_output"),
    ],
    3: [
        Path("/kaggle/input/datasets/elisevo/celebdfv1-level-3/processed_output"),
        Path("/kaggle/input/celebdfv1-level-3/processed_output"),
    ],
    4: [
        Path("/kaggle/input/datasets/vohoanghoavien/celebdfv1-level-4/processed_output"),
        Path("/kaggle/input/celebdfv1-level-4/processed_output"),
    ],
    5: [
        Path("/kaggle/input/datasets/vohoanghoavien/celebdfv1-level-5/processed_output"),
        Path("/kaggle/input/celebdfv1-level-5/processed_output"),
    ],
}


def resolve_level_root(level: int) -> Path:
    for candidate in LEVEL_DATASET_ROOTS[level]:
        if candidate.exists():
            return candidate
    candidates = "\n".join(str(p) for p in LEVEL_DATASET_ROOTS[level])
    raise FileNotFoundError(f"Không tìm thấy dataset root cho level {level}:\n{candidates}")


def build_transform_roots(level: int) -> dict[str, Path]:
    level_root = resolve_level_root(level)
    return {
        "color_contrast": level_root / "color_contrast" / f"level_{level}" / "Celeb-DF-v1",
        "color_saturation": level_root / "color_saturation" / f"level_{level}" / "Celeb-DF-v1",
        "gaussian_blur": level_root / "gaussian_blur" / f"level_{level}" / "Celeb-DF-v1",
        "resize": level_root / "resize" / f"level_{level}" / "Celeb-DF-v1",
    }

# Chọn 1 trong: sequential, shuffle, balanced_16_16
ORDER_SETTING = "sequential"
SHUFFLE_SEED = 42
BALANCED_REAL_PER_BLOCK = 16
BALANCED_FAKE_PER_BLOCK = 16

OUTPUT_DIR = Path(f"/kaggle/working/celebdfv1_{ORDER_SETTING}_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Speed knobs. Nếu Kaggle bị OOM thì giảm BATCH_SIZE xuống 64.
BATCH_SIZE = 128
NUM_WORKERS = 4
PREFETCH_FACTOR = 4
PERSISTENT_WORKERS = NUM_WORKERS > 0
SKIP_EXISTING = True
USE_AMP = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DEVICE

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df["datasetname"] == "Celeb-DF-v1"].copy()
df["label_num"] = df["label"].map({"REAL": 0, "FAKE": 1}).astype(int)
df["imagepath_fixed"] = df["imagepath"].astype(str).str.replace(
    "../input/deepfakebench",
    DEEPFAKEBENCH_ROOT,
    regex=False,
)
df = df.reset_index(drop=True)

print("raw celebdfv1 shape:", df.shape)
print(df["label_num"].value_counts().rename(index={0: "REAL", 1: "FAKE"}))
df.head()

In [ ]:
def apply_order_setting(dataframe: pd.DataFrame) -> pd.DataFrame:
    if ORDER_SETTING == "sequential":
        ordered = dataframe.copy()

    elif ORDER_SETTING == "shuffle":
        ordered = dataframe.sample(frac=1.0, random_state=SHUFFLE_SEED).copy()

    elif ORDER_SETTING == "balanced_16_16":
        real_df = dataframe[dataframe["label_num"] == 0].copy().reset_index(drop=True)
        fake_df = dataframe[dataframe["label_num"] == 1].copy().reset_index(drop=True)

        blocks = []
        real_pos = 0
        fake_pos = 0

        # Dừng khi không còn đủ 16 REAL. FAKE thường nhiều hơn nên phần dư FAKE bị bỏ.
        while real_pos + BALANCED_REAL_PER_BLOCK <= len(real_df):
            if fake_pos + BALANCED_FAKE_PER_BLOCK > len(fake_df):
                break

            blocks.append(real_df.iloc[real_pos:real_pos + BALANCED_REAL_PER_BLOCK])
            blocks.append(fake_df.iloc[fake_pos:fake_pos + BALANCED_FAKE_PER_BLOCK])

            real_pos += BALANCED_REAL_PER_BLOCK
            fake_pos += BALANCED_FAKE_PER_BLOCK

        if not blocks:
            raise ValueError("balanced_16_16 không tạo được block nào. Check label/dataframe.")

        ordered = pd.concat(blocks, axis=0)

    else:
        raise ValueError(f"Unknown ORDER_SETTING: {ORDER_SETTING}")

    return ordered.reset_index(drop=True)


ordered_df = apply_order_setting(df)
print("order setting:", ORDER_SETTING)
print("ordered shape:", ordered_df.shape)
print(ordered_df["label_num"].value_counts().rename(index={0: "REAL", 1: "FAKE"}))
ordered_df[["label", "label_num", "imagepath_fixed"]].head(40)

In [ ]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="openai",
    device=DEVICE,
)
clip_model.eval()

# Tránh update grad state và giảm overhead khi inference.
for param in clip_model.parameters():
    param.requires_grad_(False)

clip_model;

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["imagepath_fixed"]).convert("RGB")
        image = preprocess(image)
        label = int(row["label_num"])
        path = row["imagepath_fixed"]
        return image, label, path


def extract_features(dataframe: pd.DataFrame):
    loader = DataLoader(
        ImageDataset(dataframe),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    features = []
    labels = []
    paths = []

    with torch.no_grad():
        for images, batch_labels, batch_paths in tqdm(loader):
            images = images.to(DEVICE, non_blocking=True)
            batch_features = clip_model.encode_image(images)
            batch_features = F.normalize(batch_features, dim=-1)

            features.append(batch_features.cpu())
            labels.append(batch_labels.cpu())
            paths.extend(batch_paths)

    return torch.cat(features), torch.cat(labels), paths

In [ ]:
saved_paths = []

for level in LEVELS:
    transform_roots = build_transform_roots(level)
    level_output_dir = OUTPUT_DIR / f"level_{level}"
    level_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n===== LEVEL {level} =====")
    print("resolved root:", resolve_level_root(level))

    for transform_name, transform_root in transform_roots.items():
        transform_df = ordered_df.copy()
        transform_df["imagepath_fixed"] = transform_df["imagepath_fixed"].str.replace(
            CELEBDFV1_ORIGINAL_ROOT,
            str(transform_root),
            regex=False,
        )

        output_path = level_output_dir / f"celebdfv1_level{level}_{transform_name}_{ORDER_SETTING}_features.pt"

        if SKIP_EXISTING and output_path.exists():
            print(f"skip existing: {output_path}")
            saved_paths.append(output_path)
            continue

        print(f"\nExtracting level={level} transform={transform_name}")
        print("root:", transform_root)
        print("output:", output_path)

        features, labels, paths = extract_features(transform_df)

        torch.save(
            {
                "features": features,
                "labels": labels,
                "paths": paths,
                "dataset_name": "Celeb-DF-v1",
                "transform_name": transform_name,
                "transform_level": level,
                "order_setting": ORDER_SETTING,
                "shuffle_seed": SHUFFLE_SEED if ORDER_SETTING == "shuffle" else None,
                "balanced_real_per_block": BALANCED_REAL_PER_BLOCK if ORDER_SETTING == "balanced_16_16" else None,
                "balanced_fake_per_block": BALANCED_FAKE_PER_BLOCK if ORDER_SETTING == "balanced_16_16" else None,
                "clip_model": "ViT-L-14/openai",
            },
            output_path,
        )

        print("features:", features.shape)
        print("labels:", labels.shape)
        print("label counts [REAL, FAKE]:", torch.bincount(labels.long(), minlength=2).tolist())
        print("saved to:", output_path)
        saved_paths.append(output_path)

saved_paths